# Notebook 03 — Exploratory movement analysis

This notebook addresses: **How do GPS-derived location and movement patterns vary by time of day and individual in the publicly available 2017 sample?** It uses the persisted output from Notebook 02 and does not modify the source Parquet or raw GPS file.

A GPS fix is one recorded location. A movement step is the geodesic displacement between two consecutive fixes from the same individual. Because sampling is irregular, steps spanning a long tracking gap are excluded from the primary movement-step table and preserved in a separate exclusion table. The resulting average speed is displacement divided by elapsed time; it is not an instantaneous flight speed.

## 1. Introduction, provenance, and limitations

The selected Zenodo record (`10.5281/zenodo.20706447`, CC0) contains GPS telemetry from lesser black-backed gulls (*Larus fuscus*) in the LBBG_SCHIERMONNIKOOG study. The public 2017 sample contains 441,979 fixes from 20 individuals. The metadata documents UTC/GPS timestamps, WGS84 coordinates, GPS quality fields, and source ground speed, but it does not document one fixed sampling interval. Sampling frequency therefore must be measured before movement steps are interpreted.

This is an exploratory analysis. The records are repeated observations within individuals, the available public period is incomplete, and UTC hour is not local solar time. No stopovers, causal environmental effects, or migration routes are inferred here.

In [1]:
from pathlib import Path
import json
import platform
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyarrow
import seaborn as sns
from IPython.display import Markdown, display

from bird_migration.movement import (
    add_review_flags,
    derive_step_candidates,
    evaluate_gap_thresholds,
    sampling_intervals,
    sort_fixes,
)

PROJECT_ROOT = next((candidate for candidate in (Path.cwd(), *Path.cwd().parents) if (candidate / 'pyproject.toml').exists() and (candidate / 'data' / 'processed' / 'lbbg_gps_fixes_preprocessed.parquet').exists()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError(f'Could not locate repository root from working directory: {Path.cwd()}')
INPUT_PATH = PROJECT_ROOT / 'data' / 'processed' / 'lbbg_gps_fixes_preprocessed.parquet'
RAW_DIR = PROJECT_ROOT / 'data' / 'raw'
INTERIM_DIR = PROJECT_ROOT / 'data' / 'interim'
REPORT_DIR = PROJECT_ROOT / 'reports'
FIGURE_DIR = REPORT_DIR / 'figures'
for directory in (INTERIM_DIR, REPORT_DIR, FIGURE_DIR):
    directory.mkdir(parents=True, exist_ok=True)

print('Python:', sys.version.split()[0])
print('Platform:', platform.platform())
print('pandas:', pd.__version__, '| numpy:', np.__version__, '| pyarrow:', pyarrow.__version__)
print('Input:', INPUT_PATH)


Python: 3.12.12
Platform: macOS-15.6-arm64-arm-64bit
pandas: 2.3.3 | numpy: 2.5.4 | pyarrow: 19.0.1
Input: /Users/rizqititing/Documents/projects/bird-migration/data/processed/lbbg_gps_fixes_preprocessed.parquet


## 2. Load and validate the preprocessed data

In [2]:
if not INPUT_PATH.exists():
    raise FileNotFoundError(f'Notebook 02 output not found: {INPUT_PATH}')
fixes = pd.read_parquet(INPUT_PATH, engine='pyarrow')
metadata_path = RAW_DIR / 'datapackage.json'
reference_path = RAW_DIR / 'LBBG_SCHIERMONNIKOOG-reference-data.csv'
metadata = json.loads(metadata_path.read_text(encoding='utf-8')) if metadata_path.exists() else {}
reference = pd.read_csv(reference_path) if reference_path.exists() else pd.DataFrame()

required = {'individual_id', 'event_id', 'timestamp_utc', 'latitude', 'longitude'}
missing = required.difference(fixes.columns)
if missing:
    raise KeyError(f'Preprocessed Parquet is missing required columns: {sorted(missing)}')
if not pd.api.types.is_datetime64tz_dtype(fixes['timestamp_utc']):
    raise TypeError('timestamp_utc must be timezone-aware UTC timestamps')
assert len(fixes) == 441_979
assert fixes['individual_id'].nunique() == 20
assert fixes['event_id'].is_unique
assert fixes['timestamp_utc'].notna().all()
assert fixes['latitude'].between(-90, 90).all()
assert fixes['longitude'].between(-180, 180).all()
assert set(fixes['sensor_type'].dropna().unique()) == {'gps'}

gps_resource = next((r for r in metadata.get('resources', []) if r.get('name') == 'gps'), {})
print('Fixes:', len(fixes), '| Individuals:', fixes['individual_id'].nunique())
print('UTC coverage:', fixes['timestamp_utc'].min(), 'to', fixes['timestamp_utc'].max())
print('Reference deployments:', len(reference), '| GPS resource paths:', gps_resource.get('path', []))
display(fixes[['individual_id', 'timestamp_utc', 'latitude', 'longitude', 'sensor_type']].head())


Fixes: 441979 | Individuals: 20
UTC coverage: 2017-05-24 09:46:47+00:00 to 2017-12-31 23:58:28+00:00
Reference deployments: 29 | GPS resource paths: ['LBBG_SCHIERMONNIKOOG-gps-2017.csv.gz', 'LBBG_SCHIERMONNIKOOG-gps-2018.csv.gz', 'LBBG_SCHIERMONNIKOOG-gps-2019.csv.gz', 'LBBG_SCHIERMONNIKOOG-gps-2020.csv.gz']


/var/folders/78/500yhrl903d8mz5qxg9pdz6r0000gn/T/ipykernel_28941/1928192694.py:13: DeprecationWarning: is_datetime64tz_dtype is deprecated and will be removed in a future version. Check `isinstance(dtype, pd.DatetimeTZDtype)` instead.
  if not pd.api.types.is_datetime64tz_dtype(fixes['timestamp_utc']):


,individual_id,timestamp_utc,latitude,longitude,sensor_type
0,5500276,2017-05-24 09:46:47+00:00,53.499042,6.262858,gps
1,5500276,2017-05-24 09:56:43+00:00,53.506315,6.267735,gps
2,5500276,2017-05-24 10:07:09+00:00,53.506317,6.267793,gps
3,5500276,2017-05-24 10:16:59+00:00,53.506323,6.267771,gps
4,5500276,2017-05-24 10:26:51+00:00,53.506330,6.267738,gps


## 3. GPS sampling interval analysis

Intervals are calculated only between consecutive fixes within the same individual. The primary threshold is selected from the observed distribution rather than assumed regular sampling: most intervals cluster near 5–10 minutes, the 99th percentile is around 30 minutes, and the tail contains much longer outages. A one-hour threshold is used as a conservative continuity rule: it retains the dense sampling regime while excluding intervals that are plausibly tracking gaps. Sensitivity thresholds are reported alongside it.

In [3]:
ordered_fixes = sort_fixes(fixes)
intervals = sampling_intervals(ordered_fixes)
positive_intervals = intervals.loc[intervals['elapsed_time_seconds'] > 0].copy()
if len(positive_intervals) != len(fixes) - fixes['individual_id'].nunique():
    raise ValueError('Unexpected number of positive within-individual intervals')

interval_summary = positive_intervals.groupby('individual_id')['elapsed_time_seconds'].agg(
    interval_count='count', median_seconds='median', mean_seconds='mean', std_seconds='std',
    min_seconds='min', max_seconds='max',
)
interval_summary['q05_seconds'] = positive_intervals.groupby('individual_id')['elapsed_time_seconds'].quantile(0.05)
interval_summary['q25_seconds'] = positive_intervals.groupby('individual_id')['elapsed_time_seconds'].quantile(0.25)
interval_summary['q75_seconds'] = positive_intervals.groupby('individual_id')['elapsed_time_seconds'].quantile(0.75)
interval_summary['q95_seconds'] = positive_intervals.groupby('individual_id')['elapsed_time_seconds'].quantile(0.95)
interval_summary['q99_seconds'] = positive_intervals.groupby('individual_id')['elapsed_time_seconds'].quantile(0.99)
interval_summary = interval_summary.reset_index()
interval_summary.to_csv(REPORT_DIR / 'sampling_interval_summary.csv', index=False)

thresholds = evaluate_gap_thresholds(positive_intervals['elapsed_time_seconds'], [600, 900, 1800, 3600, 7200, 21600, 86400])
thresholds.to_csv(REPORT_DIR / 'sampling_gap_threshold_sensitivity.csv', index=False)
GAP_THRESHOLD_SECONDS = 3600
print('Overall interval quantiles (seconds):')
display(positive_intervals['elapsed_time_seconds'].describe(percentiles=[.01, .05, .25, .5, .75, .9, .95, .99, .999]).to_frame().T)
print('Threshold sensitivity:')
display(thresholds)
print('Selected threshold:', GAP_THRESHOLD_SECONDS, 'seconds (1 hour)')


Overall interval quantiles (seconds):


,count,mean,std,min,1%,5%,25%,50%,75%,90%,95%,99%,99.9%,max
elapsed_time_seconds,441959.0,630.926541,11902.567268,4.0,4.0,287.0,314.0,592.0,600.0,618.0,1197.0,1811.0,1852.0,4716150.0


Threshold sensitivity:


,threshold_seconds,threshold_hours,positive_intervals,retained_steps,excluded_long_gap_steps,retained_fraction
0,600,0.166667,441959,333553,108406,0.754715
1,900,0.250000,441959,416695,25264,0.942836
2,1800,0.500000,441959,433804,8155,0.981548
3,3600,1.000000,441959,441814,145,0.999672
4,7200,2.000000,441959,441944,15,0.999966
5,21600,6.000000,441959,441946,13,0.999971
6,86400,24.000000,441959,441946,13,0.999971


Selected threshold: 3600 seconds (1 hour)


In [4]:
sns.set_theme(style='whitegrid', context='notebook')
fig, ax = plt.subplots(figsize=(9, 5))
ax.hist(positive_intervals['elapsed_time_seconds'], bins=np.logspace(np.log10(1), np.log10(positive_intervals['elapsed_time_seconds'].max()), 80), color='#35618d')
ax.set_xscale('log'); ax.axvline(GAP_THRESHOLD_SECONDS, color='#b33a3a', linestyle='--', label='Selected threshold: 1 hour')
ax.set(title='Distribution of positive GPS sampling intervals', xlabel='Elapsed time between fixes (seconds, log scale)', ylabel='Number of intervals'); ax.legend()
fig.tight_layout(); fig.savefig(FIGURE_DIR / '03_sampling_interval_distribution.png', dpi=180); plt.close(fig)

fig, ax = plt.subplots(figsize=(11, 7))
sns.boxplot(data=positive_intervals, y='individual_id', x='elapsed_time_seconds', order=interval_summary.sort_values('median_seconds')['individual_id'], showfliers=False, color='#79a6c2', ax=ax)
ax.set_xscale('log'); ax.set(title='Sampling intervals by individual', xlabel='Elapsed time between fixes (seconds, log scale)', ylabel='Individual ID')
fig.tight_layout(); fig.savefig(FIGURE_DIR / '03_sampling_interval_by_individual.png', dpi=180); plt.close(fig)

positive_intervals['interval_date'] = positive_intervals['timestamp_utc'].dt.floor('D')
interval_daily = positive_intervals.groupby('interval_date')['elapsed_time_seconds'].median().reset_index()
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(interval_daily['interval_date'], interval_daily['elapsed_time_seconds'], color='#2a7f62', linewidth=1)
ax.axhline(GAP_THRESHOLD_SECONDS, color='#b33a3a', linestyle='--', linewidth=1)
ax.set(title='Median sampling interval over time (UTC date)', xlabel='UTC date', ylabel='Median interval (seconds)')
fig.tight_layout(); fig.savefig(FIGURE_DIR / '03_sampling_interval_over_time.png', dpi=180); plt.close(fig)


## 4. Derive geodesic movement steps

Consecutive coordinates are connected only within individual and only when timestamps and coordinates are valid. Distances use `pyproj.Geod` on the WGS84 ellipsoid, not Euclidean degrees. Long-gap pairs are preserved as excluded candidates and are not bridged in the primary step table. Source `ground-speed` is retained separately from GPS-derived average speed.

In [5]:
step_candidates = derive_step_candidates(fixes, GAP_THRESHOLD_SECONDS)
retained_steps = step_candidates.loc[step_candidates['step_retained']].copy()
excluded_steps = step_candidates.loc[~step_candidates['step_retained']].copy()
retained_steps, review_thresholds = add_review_flags(retained_steps)

assert len(step_candidates) == len(fixes)
assert len(retained_steps) + len(excluded_steps) == len(fixes)
assert retained_steps['same_individual'].all()
assert (retained_steps['elapsed_time_seconds'] > 0).all()
assert (retained_steps['step_distance_m'] >= 0).all()
assert np.allclose(retained_steps['average_speed_m_s'], retained_steps['step_distance_m'] / retained_steps['elapsed_time_seconds'])

excluded_steps.to_parquet(INTERIM_DIR / 'lbbg_movement_steps_excluded.parquet', engine='pyarrow', index=False)
excluded_steps.to_csv(REPORT_DIR / 'movement_step_exclusions.csv', index=False)
retained_steps.to_parquet(INTERIM_DIR / 'lbbg_movement_steps.parquet', engine='pyarrow', index=False)

print('Candidate pair rows:', len(step_candidates))
print('Retained movement steps:', len(retained_steps))
print('Excluded candidate rows:', len(excluded_steps))
print('Exclusion reasons:')
display(excluded_steps['step_exclusion_reason'].value_counts().rename_axis('reason').reset_index(name='count'))
print('Review thresholds:', review_thresholds)


Candidate pair rows: 441979
Retained movement steps: 441814
Excluded candidate rows: 165
Exclusion reasons:


,reason,count
0,gap_exceeds_threshold,145
1,no_next_fix,20


Review thresholds: {'distance_review_quantile': 0.999, 'distance_review_threshold_m': 13182.238798575418, 'speed_review_quantile': 0.999, 'speed_review_threshold_m_s': 18.38072978191438}


## 5. Movement quality control

Review flags are transparent screening indicators, not biological or measurement-error verdicts. The distance and speed thresholds are the 99.9th percentiles among retained steps; GPS DOP > 5 and fewer than 4 satellites are conventional quality-review heuristics. No flagged retained step is removed.

In [6]:
quality_rows = [
    {'metric': 'positive_intervals', 'value': int(len(positive_intervals)), 'definition': 'Consecutive within-individual intervals greater than zero.'},
    {'metric': 'retained_steps', 'value': int(len(retained_steps)), 'definition': 'Valid within-individual pairs with elapsed time <= 1 hour.'},
    {'metric': 'excluded_no_next_fix', 'value': int((excluded_steps['step_exclusion_reason'] == 'no_next_fix').sum()), 'definition': 'Terminal fix for each individual; no next fix exists.'},
    {'metric': 'excluded_long_gap', 'value': int((excluded_steps['step_exclusion_reason'] == 'gap_exceeds_threshold').sum()), 'definition': 'Positive elapsed time greater than 1 hour.'},
    {'metric': 'excluded_invalid_time', 'value': int((excluded_steps['step_exclusion_reason'] == 'nonpositive_or_invalid_elapsed_time').sum()), 'definition': 'Nonpositive or invalid elapsed time.'},
    {'metric': 'excluded_invalid_coordinates', 'value': int((excluded_steps['step_exclusion_reason'] == 'invalid_coordinate_pair').sum()), 'definition': 'Coordinate outside WGS84 bounds or missing.'},
    {'metric': 'large_distance_review_flags', 'value': int(retained_steps['large_distance_review_flag'].sum()), 'definition': f"Step distance above retained {review_thresholds['distance_review_quantile']:.3f} quantile ({review_thresholds['distance_review_threshold_m']:.2f} m)."},
    {'metric': 'high_speed_review_flags', 'value': int(retained_steps['high_speed_review_flag'].sum()), 'definition': f"Derived average speed above retained {review_thresholds['speed_review_quantile']:.3f} quantile ({review_thresholds['speed_review_threshold_m_s']:.2f} m/s)."},
    {'metric': 'gps_dop_review_flags', 'value': int(retained_steps['gps_dop_review_flag'].sum()), 'definition': 'Start-fix GPS DOP greater than 5.'},
    {'metric': 'low_satellite_review_flags', 'value': int(retained_steps['low_satellite_count_review_flag'].sum()), 'definition': 'Start-fix satellite count below 4.'},
    {'metric': 'any_review_flag', 'value': int(retained_steps['any_review_flag'].sum()), 'definition': 'At least one transparent review flag.'},
]
quality_report = pd.DataFrame(quality_rows)
quality_report.to_csv(REPORT_DIR / 'movement_quality_report.csv', index=False)
display(quality_report)

source_speed = retained_steps[['source_ground_speed_m_s', 'average_speed_m_s']].dropna()
speed_comparison = pd.DataFrame([
    {'metric': 'paired_records', 'value': len(source_speed)},
    {'metric': 'source_ground_speed_median_m_s', 'value': source_speed['source_ground_speed_m_s'].median()},
    {'metric': 'derived_average_speed_median_m_s', 'value': source_speed['average_speed_m_s'].median()},
    {'metric': 'median_derived_minus_source_m_s', 'value': (source_speed['average_speed_m_s'] - source_speed['source_ground_speed_m_s']).median()},
    {'metric': 'pearson_correlation', 'value': source_speed.corr().loc['source_ground_speed_m_s', 'average_speed_m_s']},
])
speed_comparison.to_csv(REPORT_DIR / 'source_vs_derived_speed_comparison.csv', index=False)
display(speed_comparison)


,metric,value,definition
0,positive_intervals,441959,Consecutive within-individual intervals greate...
1,retained_steps,441814,Valid within-individual pairs with elapsed tim...
2,excluded_no_next_fix,20,Terminal fix for each individual; no next fix ...
3,excluded_long_gap,145,Positive elapsed time greater than 1 hour.
4,excluded_invalid_time,0,Nonpositive or invalid elapsed time.
5,excluded_invalid_coordinates,0,Coordinate outside WGS84 bounds or missing.
6,large_distance_review_flags,442,Step distance above retained 0.999 quantile (1...
7,high_speed_review_flags,442,Derived average speed above retained 0.999 qua...
8,gps_dop_review_flags,29691,Start-fix GPS DOP greater than 5.
9,low_satellite_review_flags,0,Start-fix satellite count below 4.


,metric,value
0,paired_records,439290.000000
1,source_ground_speed_median_m_s,0.745550
2,derived_average_speed_median_m_s,0.023355
3,median_derived_minus_source_m_s,-0.574428
4,pearson_correlation,0.665332


## 6. Exploratory movement visualisations

In [7]:
retained_steps['start_utc_hour'] = retained_steps['start_timestamp'].dt.hour
retained_steps['start_utc_date'] = retained_steps['start_timestamp'].dt.floor('D')
retained_steps['start_utc_month'] = retained_steps['start_timestamp'].dt.month

fig, ax = plt.subplots(figsize=(9, 5))
ax.hist(retained_steps['step_distance_m'], bins=80, color='#35618d')
ax.set(title='Retained movement-step distance distribution', xlabel='Geodesic step distance (m)', ylabel='Number of steps')
fig.tight_layout(); fig.savefig(FIGURE_DIR / '03_step_distance_distribution.png', dpi=180); plt.close(fig)

fig, ax = plt.subplots(figsize=(9, 5))
ax.hist(retained_steps['average_speed_m_s'], bins=80, color='#2a7f62')
ax.set(title='Derived average-speed distribution', xlabel='Average displacement speed (m/s)', ylabel='Number of steps')
fig.tight_layout(); fig.savefig(FIGURE_DIR / '03_derived_average_speed_distribution.png', dpi=180); plt.close(fig)

hourly = retained_steps.groupby('start_utc_hour').agg(mean_distance_m=('step_distance_m', 'mean'), median_distance_m=('step_distance_m', 'median'), mean_speed_m_s=('average_speed_m_s', 'mean'), median_speed_m_s=('average_speed_m_s', 'median'), step_count=('step_distance_m', 'size')).reset_index()
hourly.to_csv(REPORT_DIR / 'movement_summary_by_utc_hour.csv', index=False)
fig, axes = plt.subplots(2, 1, figsize=(11, 8), sharex=True)
axes[0].plot(hourly['start_utc_hour'], hourly['mean_distance_m'], marker='o', color='#35618d'); axes[0].set(ylabel='Mean step distance (m)', title='Movement distance by UTC hour')
axes[1].plot(hourly['start_utc_hour'], hourly['mean_speed_m_s'], marker='o', color='#2a7f62'); axes[1].set(xlabel='UTC hour', ylabel='Mean derived average speed (m/s)', title='Average displacement speed by UTC hour'); axes[1].set_xticks(range(24))
fig.tight_layout(); fig.savefig(FIGURE_DIR / '03_movement_by_utc_hour.png', dpi=180); plt.close(fig)

fig, ax = plt.subplots(figsize=(11, 7))
sns.boxplot(data=retained_steps, x='individual_id', y='step_distance_m', showfliers=False, color='#79a6c2', ax=ax)
ax.tick_params(axis='x', rotation=65); ax.set(title='Movement-step distances by individual', xlabel='Individual ID', ylabel='Geodesic step distance (m)')
fig.tight_layout(); fig.savefig(FIGURE_DIR / '03_movement_by_individual.png', dpi=180); plt.close(fig)

daily = retained_steps.groupby('start_utc_date').agg(total_distance_m=('step_distance_m', 'sum'), median_step_distance_m=('step_distance_m', 'median'), mean_speed_m_s=('average_speed_m_s', 'mean'), step_count=('step_distance_m', 'size')).reset_index()
daily.to_csv(REPORT_DIR / 'movement_daily_summary.csv', index=False)
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(daily['start_utc_date'], daily['total_distance_m'] / 1000, color='#9b3d5a', linewidth=1)
ax.set(title='Daily retained movement-step distance (UTC date)', xlabel='UTC date', ylabel='Total step distance (km)')
fig.tight_layout(); fig.savefig(FIGURE_DIR / '03_daily_movement_summary.png', dpi=180); plt.close(fig)

fig, ax = plt.subplots(figsize=(9, 6))
sample = retained_steps.sample(min(100_000, len(retained_steps)), random_state=42)
ax.scatter(sample['elapsed_time_seconds'] / 60, sample['step_distance_m'], s=3, alpha=0.15, color='#35618d')
ax.set(xscale='log', yscale='log', title='Step distance and elapsed time', xlabel='Elapsed time (minutes, log scale)', ylabel='Geodesic step distance (m, log scale)')
fig.tight_layout(); fig.savefig(FIGURE_DIR / '03_distance_vs_elapsed_time.png', dpi=180); plt.close(fig)
print('Saved exploratory figures under', FIGURE_DIR)


Saved exploratory figures under /Users/rizqititing/Documents/projects/bird-migration/reports/figures


## 7. Temporal exploration and individual comparison

In [8]:
monthly = retained_steps.groupby('start_utc_month').agg(total_distance_m=('step_distance_m', 'sum'), mean_distance_m=('step_distance_m', 'mean'), mean_speed_m_s=('average_speed_m_s', 'mean'), step_count=('step_distance_m', 'size')).reset_index()
monthly.to_csv(REPORT_DIR / 'movement_summary_by_month.csv', index=False)

individual_movement = retained_steps.groupby('individual_id').agg(
    valid_movement_steps=('step_distance_m', 'size'),
    median_step_distance_m=('step_distance_m', 'median'),
    mean_step_distance_m=('step_distance_m', 'mean'),
    median_derived_average_speed_m_s=('average_speed_m_s', 'median'),
    mean_derived_average_speed_m_s=('average_speed_m_s', 'mean'),
).reset_index()
individual_movement = individual_movement.merge(interval_summary[['individual_id', 'interval_count', 'median_seconds', 'mean_seconds', 'std_seconds', 'max_seconds']], on='individual_id', how='left')
individual_movement = individual_movement.rename(columns={'interval_count': 'positive_intervals', 'median_seconds': 'median_sampling_interval_seconds', 'mean_seconds': 'mean_sampling_interval_seconds', 'std_seconds': 'sd_sampling_interval_seconds', 'max_seconds': 'maximum_sampling_interval_seconds'})
gap_counts = excluded_steps.loc[excluded_steps['step_exclusion_reason'] == 'gap_exceeds_threshold'].groupby('individual_id').size().rename('excluded_long_gap_steps')
individual_movement = individual_movement.join(gap_counts, on='individual_id').fillna({'excluded_long_gap_steps': 0})
individual_movement['percentage_steps_excluded_due_to_large_gaps'] = individual_movement['excluded_long_gap_steps'] / (individual_movement['valid_movement_steps'] + individual_movement['excluded_long_gap_steps']) * 100
individual_movement.to_csv(REPORT_DIR / 'movement_summary_by_individual.csv', index=False)
display(individual_movement)
print('UTC is used for descriptive grouping only; it is not local solar time.')


,individual_id,valid_movement_steps,median_step_distance_m,mean_step_distance_m,median_derived_average_speed_m_s,mean_derived_average_speed_m_s,positive_intervals,median_sampling_interval_seconds,mean_sampling_interval_seconds,sd_sampling_interval_seconds,maximum_sampling_interval_seconds,excluded_long_gap_steps,percentage_steps_excluded_due_to_large_gaps
0,5500276,50105,9.534292,325.697984,0.030621,0.924786,50115,288.0,382.027457,326.935819,12576.0,10.0,0.019954
1,5500277,26336,10.612923,702.297960,0.018963,1.878787,26360,593.0,726.149014,14099.467503,2288546.0,24.0,0.091047
2,5500278,8698,11.787061,664.885421,0.018874,1.069895,8698,593.0,622.159117,126.521154,2389.0,0.0,0.000000
3,5500279,26299,20.062858,310.690726,0.033236,0.508991,26301,593.0,692.591270,10237.039776,1489382.0,2.0,0.007604
4,5500281,56325,11.208897,331.391419,0.069076,1.064237,56328,288.0,339.648203,1049.703665,244120.0,3.0,0.005326
5,5500282,19447,20.002020,643.998917,0.024466,0.757529,19489,595.0,981.404998,12979.627345,1811380.0,42.0,0.215506
6,5500283,31703,8.691460,529.273926,0.014471,0.874890,31703,593.0,603.312431,60.656938,1846.0,0.0,0.000000
7,5500284,27903,16.162608,781.869841,0.023966,1.173276,27905,593.0,685.349077,306.883438,3673.0,2.0,0.007167
8,5500285,7734,7.173449,1044.703568,0.011620,1.671907,7735,593.0,628.640465,152.612383,3750.0,1.0,0.012928
9,5500286,1936,15.290201,621.691676,0.020924,0.812089,1942,602.0,798.180742,475.196370,3684.0,6.0,0.308960


UTC is used for descriptive grouping only; it is not local solar time.


## 8. Research interpretation

The interpretation below is generated from the calculated tables. It is descriptive and does not treat movement steps as independent observations.

In [9]:
hour_peak_distance = hourly.loc[hourly['mean_distance_m'].idxmax()]
hour_peak_speed = hourly.loc[hourly['mean_speed_m_s'].idxmax()]
individual_distance_range = individual_movement['median_step_distance_m'].agg(['min', 'max'])
daily_peak = daily.loc[daily['total_distance_m'].idxmax()]
interpretation = f"""
### Exploratory findings

The observed positive sampling intervals have a median of {positive_intervals['elapsed_time_seconds'].median():.1f} seconds and a 99th percentile of {positive_intervals['elapsed_time_seconds'].quantile(.99):.1f} seconds. The one-hour continuity rule retains {len(retained_steps):,} of {len(positive_intervals):,} positive intervals ({len(retained_steps) / len(positive_intervals) * 100:.2f}%) and excludes {int((excluded_steps['step_exclusion_reason'] == 'gap_exceeds_threshold').sum()):,} long-gap pairs. The remaining {int((excluded_steps['step_exclusion_reason'] == 'no_next_fix').sum())} records are terminal fixes without a next observation, not movement steps.

Among retained steps, median geodesic displacement is {retained_steps['step_distance_m'].median():.1f} m and median derived average displacement speed is {retained_steps['average_speed_m_s'].median():.2f} m/s. The largest hourly mean step distance occurs at UTC hour {int(hour_peak_distance['start_utc_hour'])}, while the largest hourly mean derived average speed occurs at UTC hour {int(hour_peak_speed['start_utc_hour'])}. These are UTC associations only and should not be interpreted as day/night effects without local solar calculations.

Individual median step distances range from {individual_distance_range['min']:.1f} to {individual_distance_range['max']:.1f} m. The day with the greatest retained summed step distance is {pd.Timestamp(daily_peak['start_utc_date']).date()}, with {daily_peak['total_distance_m'] / 1000:.2f} km summed across retained steps. These totals reflect both movement and observation coverage.

### Limitations and uncertainty

Sampling is irregular and individual coverage differs. Long gaps may represent missing transmission, tag scheduling, or unobserved movement; excluding them reduces artificial bridging but does not remove all sampling bias. Geodesic displacement divided by elapsed time is an interval-average speed, not instantaneous flight speed. Review flags identify unusual values for inspection and do not establish measurement error. Repeated steps within individuals require individual-aware statistical models in later work. The public sample covers only the available 2017 records, and no stopover or causal environmental conclusion is supported by this exploratory notebook.
"""
display(Markdown(interpretation))
(REPORT_DIR / 'movement_interpretation.txt').write_text(interpretation, encoding='utf-8')



### Exploratory findings

The observed positive sampling intervals have a median of 592.0 seconds and a 99th percentile of 1811.0 seconds. The one-hour continuity rule retains 441,814 of 441,959 positive intervals (99.97%) and excludes 145 long-gap pairs. The remaining 20 records are terminal fixes without a next observation, not movement steps.

Among retained steps, median geodesic displacement is 11.4 m and median derived average displacement speed is 0.02 m/s. The largest hourly mean step distance occurs at UTC hour 4, while the largest hourly mean derived average speed occurs at UTC hour 4. These are UTC associations only and should not be interpreted as day/night effects without local solar calculations.

Individual median step distances range from 6.0 to 20.1 m. The day with the greatest retained summed step distance is 2017-06-20, with 3391.59 km summed across retained steps. These totals reflect both movement and observation coverage.

### Limitations and uncertainty

Sampling is irregular and individual coverage differs. Long gaps may represent missing transmission, tag scheduling, or unobserved movement; excluding them reduces artificial bridging but does not remove all sampling bias. Geodesic displacement divided by elapsed time is an interval-average speed, not instantaneous flight speed. Review flags identify unusual values for inspection and do not establish measurement error. Repeated steps within individuals require individual-aware statistical models in later work. The public sample covers only the available 2017 records, and no stopover or causal environmental conclusion is supported by this exploratory notebook.


1661

## 9. Export and final validation

In [10]:
reloaded_steps = pd.read_parquet(INTERIM_DIR / 'lbbg_movement_steps.parquet', engine='pyarrow')
reloaded_excluded = pd.read_parquet(INTERIM_DIR / 'lbbg_movement_steps_excluded.parquet', engine='pyarrow')
assert len(reloaded_steps) == len(retained_steps)
assert reloaded_steps['individual_id'].notna().all()
assert reloaded_steps['elapsed_time_seconds'].gt(0).all()
assert reloaded_steps['step_distance_m'].ge(0).all()
assert np.allclose(reloaded_steps['average_speed_m_s'], reloaded_steps['step_distance_m'] / reloaded_steps['elapsed_time_seconds'])
assert reloaded_steps['same_individual'].all()
assert not reloaded_steps['sampling_gap_flag'].any()
assert len(reloaded_excluded) == len(excluded_steps)
assert set(reloaded_steps['individual_id'].unique()) == set(fixes['individual_id'].unique())

expected_outputs = [
    INTERIM_DIR / 'lbbg_movement_steps.parquet',
    INTERIM_DIR / 'lbbg_movement_steps_excluded.parquet',
    REPORT_DIR / 'movement_summary_by_individual.csv',
    REPORT_DIR / 'sampling_interval_summary.csv',
    REPORT_DIR / 'movement_quality_report.csv',
    REPORT_DIR / 'movement_step_exclusions.csv',
    FIGURE_DIR / '03_sampling_interval_distribution.png',
    FIGURE_DIR / '03_sampling_interval_by_individual.png',
    FIGURE_DIR / '03_sampling_interval_over_time.png',
    FIGURE_DIR / '03_step_distance_distribution.png',
    FIGURE_DIR / '03_derived_average_speed_distribution.png',
    FIGURE_DIR / '03_movement_by_utc_hour.png',
    FIGURE_DIR / '03_movement_by_individual.png',
    FIGURE_DIR / '03_daily_movement_summary.png',
    FIGURE_DIR / '03_distance_vs_elapsed_time.png',
]
missing_outputs = [str(path) for path in expected_outputs if not path.exists()]
if missing_outputs:
    raise FileNotFoundError(f'Missing expected Notebook 03 outputs: {missing_outputs}')
print('Final validation passed')
print('GPS fixes processed:', len(fixes))
print('Movement steps derived:', len(reloaded_steps))
print('Excluded candidate rows:', len(reloaded_excluded))
print('Excluded reasons:', reloaded_excluded['step_exclusion_reason'].value_counts().to_dict())
print('Outputs verified:', len(expected_outputs))


Final validation passed
GPS fixes processed: 441979
Movement steps derived: 441814
Excluded candidate rows: 165
Excluded reasons: {'gap_exceeds_threshold': 145, 'no_next_fix': 20}
Outputs verified: 15
